# 03 — Descriptive Indicators and Age-Sex Profile

This notebook takes the validated 85-row LGU-year panel from notebook 02 and produces the
descriptive layer of the study:

1. Incidence per 100,000 population for every LGU-year.
2. Year-over-year percent change in incidence, computed within each LGU.
3. Each LGU's own five-year average incidence, and its 2025 value against that average.
4. The NCR-level age and sex profile, which is descriptive only and is excluded from the regression.
5. Calculation validation round 2 — hand-recomputing a sample of the indicators above from the raw
   source files, independently of the pipeline code.

All indicator logic lives in `src/indicators.py` and `src/age_sex.py`, and is covered by
`tests/test_indicators.py` and `tests/test_age_sex.py` (`pytest -v`).

Nothing here is a forecast. Every figure describes a year that has already been observed.

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make src/ importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

from src.indicators import (
    add_incidence_rate,
    compute_incidence_rate,
    compute_yoy_change,
    compute_five_year_average,
    compute_current_vs_average,
    build_indicator_panel,
)
from src.age_sex import (
    load_age_sex_tables,
    build_age_summary,
    build_sex_summary,
    build_pooled_age_profile,
    summarise_child_share,
)

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "01_raw"
REF = ROOT / "data" / "02_official_reference"
VALIDATED = ROOT / "data" / "04_validated"
ANALYTICAL = ROOT / "outputs" / "analytical"
EXPORTS = ROOT / "outputs" / "dashboard_exports"

ANALYTICAL.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 1. Load the validated panel

The panel is read from `data/04_validated/lgu_year_panel.csv`, the output of notebook 02. Nothing
in this notebook edits case counts or population figures; it only derives indicators from them.

In [ ]:
panel = pd.read_csv(VALIDATED / "lgu_year_panel.csv")

assert len(panel) == 85, "Expected the 85-row LGU-year panel"
assert panel["LGU"].nunique() == 17

print(f"Panel: {panel.shape[0]} rows x {panel.shape[1]} columns")
panel.head()

## 2. Incidence per 100,000 population

Raw case counts cannot be compared across LGUs: Quezon City has roughly 45 times the population of
Pateros, so it will report more cases at any level of risk. Dividing by population and scaling to a
common denominator of 100,000 puts every LGU on the same footing.

```
incidence = dengue cases / population * 100,000
```

The population used is that LGU-year's own estimate, so an interpolated or extrapolated population
carries its uncertainty into the incidence figure. That is why the `Status` column travels with
every row.

The incidence rate is a **descriptive indicator**. The regression in notebook 04 is fitted on raw
counts with a log-population offset, not on this rate — the offset achieves the same
population adjustment while keeping the count distribution the model needs.

`compute_incidence_rate()` is the single definition of this formula in the project.
`src/panel.py` calls it too, so the column already stored in the validated panel and any column
computed here come from the same function.

In [ ]:
# The stored column and a fresh computation must be identical, since both come
# from compute_incidence_rate().
recomputed = compute_incidence_rate(panel["Dengue Cases"], panel["Population"])
largest_gap = (panel["Incidence Rate"] - recomputed).abs().max()

print(f"Largest difference between stored and recomputed incidence: {largest_gap:.2e}")
assert largest_gap < 1e-9

panel[["LGU", "Year", "Dengue Cases", "Population", "Incidence Rate", "Status"]].head(10)

In [ ]:
# Distribution of incidence by year, for the descriptive write-up
panel.groupby("Year")["Incidence Rate"].describe()[["mean", "min", "50%", "max"]].round(2)

## 3. Unit test: the incidence function

`tests/test_indicators.py` covers the formula directly rather than only its output: a known case
(100 cases in 200,000 people is 50 per 100,000), zero cases, scale invariance (doubling both cases
and population leaves the rate unchanged), rejection of a zero or negative population, rejection of
negative case counts, and agreement with the stored column across all 85 rows.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_indicators.py", "-k", "incidence"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout[-2500:])

## 4. Year-over-year percent change

```
YoY % change = (this year's incidence - last year's incidence) / last year's incidence * 100
```

The comparison is always against the **same LGU's** previous year. The panel is sorted by LGU and
then year before the shift, so the previous value can never leak in from the LGU above it in the
file. 2021 is the first year in the panel and has no prior year, so its change is left empty rather
than filled with a zero, which would read as no change.

In [ ]:
panel_yoy = compute_yoy_change(panel)

# One LGU shown end to end, so the arithmetic can be followed by eye
panel_yoy[panel_yoy["LGU"] == "Caloocan"][
    ["LGU", "Year", "Incidence Rate", "Previous Year Incidence", "YoY % Change"]
].round(2)

In [ ]:
# Region-wide shape of the five years
panel_yoy.groupby("Year")["YoY % Change"].agg(["mean", "min", "max"]).round(1)

## 5. Five-year average and the current year against it

Each LGU's five-year average incidence is the mean of its own 2021 to 2025 values. It is a
within-LGU benchmark: it says what a typical year looks like for that LGU, not how it compares with
its neighbours.

The 2025 value is then read against that benchmark. `At or Above Average` is the second of the two
conditions in the study's three-tier priority rule:

- **Priority** — in the top third of the structural risk ranking **and** 2025 incidence at or above
  the LGU's own five-year average
- **Watch** — one condition met
- **Stable** — neither

Only the second condition is computed here. The structural ranking comes from the regression, so
the tiers themselves are assigned after the model is fitted.

In [ ]:
indicators = build_indicator_panel(panel)

comparison = compute_current_vs_average(indicators, current_year=2025)
print(f"{int(comparison['At or Above Average'].sum())} of 17 LGUs are at or above their own five-year average in 2025")
comparison.round(2)

## 6. Unit tests: year-over-year change and five-year average

The assertions cover a known 100 to 150 increase reading as +50%, a decline reading as negative,
the first year having no value, the change never being computed across two different LGUs, correct
results when rows arrive out of order, the average equalling the mean of the five values, an
incomplete LGU raising rather than averaging over four years, and the at-or-above flag being true on
an exact tie.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_indicators.py"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout[-3000:])

## 7. NCR-level age and sex profile

The FOI release reports age and sex breakdowns for the National Capital Region as a whole, one
table per year, with no LGU disaggregation.

They are therefore **excluded from the regression**. Every other variable in the panel is measured
at the LGU-year level, and a single regional figure repeated across 17 LGUs carries no
between-LGU information — it would enter the model as a constant, not as a predictor. They are kept
for the descriptive section of the paper and the descriptive panel of the dashboard.

Both summaries are reconciled against the LGU-level grand totals before being returned, so the
descriptive figures and the modelled figures rest on the same numbers.

In [ ]:
age_sex_tables = load_age_sex_tables(str(RAW))

age_summary = build_age_summary(age_sex_tables)
sex_summary = build_sex_summary(age_sex_tables)
pooled_age = build_pooled_age_profile(age_summary)
child_share = summarise_child_share(age_summary)

print("Sex distribution by year")
display(sex_summary.round(2))

print("\nAge profile pooled across 2021-2025")
display(pooled_age.round(2))

print("\nShare of cases aged 0-14")
display(child_share.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(pooled_age["Age Group"], pooled_age["Total"])
ax.set_xlabel("Age group")
ax.set_ylabel("Dengue cases, 2021-2025 combined")
ax.set_title("NCR dengue cases by age group (descriptive, not used in the regression)")
plt.tight_layout()
plt.savefig(ANALYTICAL / "age_profile.png", dpi=150)
plt.show()

## 8. Unit tests: the age-sex summary

The checks confirm 13 age bands for each of the five years, per-year totals matching the FOI grand
totals, female plus male equalling the total in every row, shares summing to 100, the Grand Total
row being excluded from the body, and a deliberately altered table raising rather than passing
through.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_age_sex.py"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout[-2500:])

## 9. Calculation validation round 2 — hand-recompute a sample

Round 1 (notebook 02) validated population, density and incidence for five sampled LGU-years.
Round 2 validates the indicators built on top of them: incidence, year-over-year change, and the
five-year average.

Every hand figure below is rebuilt from the raw source CSVs with plain arithmetic, without calling
`src/indicators.py`, and then compared against the pipeline output. Full write-up:
`docs/calculation_validation_round2.md`.

In [ ]:
# Rebuild the inputs from the raw files, not from the processed panel
cases_raw = pd.read_csv(RAW / "dengue_datasets__local_govt_level.csv", thousands=",").set_index("LGU")
pop_2020 = pd.read_csv(REF / "ncr_population_masterlist__2020.csv").set_index("City and Municipality")
pop_2024 = pd.read_csv(REF / "ncr_population_masterlist__2024.csv").set_index("City and Municipality")


def source_key(lgu):
    # The raw files spell one LGU differently from the canonical panel
    return "Pasay City" if lgu == "Pasay" else lgu


def hand_population(lgu, year):
    p0 = pop_2020.loc[source_key(lgu), "Total Population"]
    p1 = pop_2024.loc[source_key(lgu), "Total Population"]
    if year == 2024:
        return float(p1)
    return p0 + (p1 - p0) * (year - 2020) / (2024 - 2020)


def hand_incidence(lgu, year):
    return cases_raw.loc[source_key(lgu), str(year)] / hand_population(lgu, year) * 100_000


samples = [
    ("Manila", 2025),        # largest LGU, extrapolated population
    ("Caloocan", 2022),      # the 2022 surge year
    ("Makati", 2023),        # a declining-population LGU
    ("Pateros", 2025),       # smallest LGU, sharp decline
    ("Malabon", 2025),       # sits just below its own average
]

rows = []
for lgu, year in samples:
    hand_ir = hand_incidence(lgu, year)
    hand_yoy = (hand_ir - hand_incidence(lgu, year - 1)) / hand_incidence(lgu, year - 1) * 100
    hand_avg = sum(hand_incidence(lgu, y) for y in range(2021, 2026)) / 5

    pipe = indicators[(indicators["LGU"] == lgu) & (indicators["Year"] == year)].iloc[0]

    rows.append({
        "LGU": lgu, "Year": year,
        "IR (hand)": round(hand_ir, 2), "IR (pipeline)": round(pipe["Incidence Rate"], 2),
        "YoY (hand)": round(hand_yoy, 2), "YoY (pipeline)": round(pipe["YoY % Change"], 2),
        "5yr avg (hand)": round(hand_avg, 2),
        "5yr avg (pipeline)": round(pipe["Five-Year Average Incidence"], 2),
    })

validation2 = pd.DataFrame(rows)
validation2["IR match"] = (validation2["IR (hand)"] - validation2["IR (pipeline)"]).abs() < 0.01
validation2["YoY match"] = (validation2["YoY (hand)"] - validation2["YoY (pipeline)"]).abs() < 0.01
validation2["Avg match"] = (validation2["5yr avg (hand)"] - validation2["5yr avg (pipeline)"]).abs() < 0.01

assert validation2[["IR match", "YoY match", "Avg match"]].all().all(), "Calculation validation round 2 FAILED"
print("Calculation validation round 2: all 5 sampled LGU-years match hand-computed values.\n")
validation2

## 10. Write the dashboard exports

Four draft CSVs are written for the Tableau workbook. They are drafts in the sense that the
priority tier column is not yet present — it is added once the regression ranking exists — but the
column names and grain are final, so the workbook built on them will not need rewiring later.

| File | Grain | Used for |
|---|---|---|
| `indicator_panel.csv` | LGU-year, 85 rows | trend lines, incidence map, YoY view |
| `lgu_summary_2025.csv` | LGU, 17 rows | current year against each LGU's own average |
| `age_profile.csv` | year and age band, 65 rows | descriptive age panel |
| `sex_profile.csv` | year, 5 rows | descriptive sex panel |

In [ ]:
indicator_columns = [
    "LGU", "Year", "Dengue Cases", "Population", "Land Area", "Population Density",
    "Incidence Rate", "Previous Year Incidence", "YoY % Change",
    "Five-Year Average Incidence", "Status",
]

indicators[indicator_columns].to_csv(EXPORTS / "indicator_panel.csv", index=False)
comparison.to_csv(EXPORTS / "lgu_summary_2025.csv", index=False)
age_summary.to_csv(EXPORTS / "age_profile.csv", index=False)
sex_summary.to_csv(EXPORTS / "sex_profile.csv", index=False)

print("Written:")
for name in ["indicator_panel.csv", "lgu_summary_2025.csv", "age_profile.csv", "sex_profile.csv"]:
    written = pd.read_csv(EXPORTS / name)
    print(f"  {name:<26} {written.shape[0]:>3} rows x {written.shape[1]} columns")

## Summary

- Incidence per 100,000 computed for all 85 LGU-years from a single tested function.
- Year-over-year change computed within each LGU; 2021 correctly carries no value.
- Five-year average computed per LGU, and 2025 compared against each LGU's own benchmark.
- Age and sex summarised at NCR level for descriptive use only, reconciled to the FOI totals.
- Calculation validation round 2 found no discrepancies: `docs/calculation_validation_round2.md`.
- Four draft CSVs exported for the Tableau workbook.

**Next:** notebook 04 fits the Poisson baseline with a log-population offset and measures the
overdispersion that motivates the Negative Binomial specification.